# Dense Retrieval and Dual Encoders (DPR)

*Narrative companion to `dense_retrieval_dual_encoders.py`. Every number here is produced by
importing that module — the notebook never redefines the math.*

InfoNCE told us how a dual encoder is **trained**. Here we ask what that architecture can
**represent** and what it **costs**, in three movements:

1. **Separability $\Rightarrow$ precomputability $\Rightarrow$ MIPS.** The score
   $s(q,d)=\langle E_Q(q),E_P(d)\rangle$ factorizes, so documents are encoded once offline and a
   query is answered by one maximum-inner-product search.
2. **The rank-$d$ expressivity ceiling.** The realizable score matrix $S=QG^\top$ has
   $\operatorname{rank}(S)\le d$; truncated SVD (Eckart–Young–Mirsky) is the best rank-$d$
   approximation, and retrieval accuracy collapses when $d$ falls below the relevance pattern's
   intrinsic rank.
3. **In-batch negatives = the $B^2$-from-$2B$ Gram trick.** A batch of $B$ pairs costs $2B$
   encoder passes but yields $B(B-1)$ negatives — and the loss is *imported* from the InfoNCE
   topic, not reimplemented.

In [ ]:
import pathlib, sys
# Author-time the module sits one level below notebooks/; importable by its underscored name.
sys.path.insert(0, str(pathlib.Path.cwd()))
sys.path.insert(0, str(pathlib.Path.cwd() / "notebooks" / "dense-retrieval-dual-encoders"))
import numpy as np
import dense_retrieval_dual_encoders as dpr

Q, P, truth, sector = dpr.dpr_finance_matrix()
S = dpr.score_matrix(Q, P)
print(f"{Q.shape[0]} queries, {P.shape[0]} documents (one per company), dim = {dpr.DPR_DIM}")
print(f"intrinsic rank(S) = {np.linalg.matrix_rank(S)} = #companies")

## Movement 1 — Separability $\Rightarrow$ precomputability $\Rightarrow$ MIPS

A dual encoder's score is a single inner product, so stacking the corpus as $P$ (one row per
document) makes top-$k$ retrieval exactly the top-$k$ of the matrix–vector product $P\,E_Q(q)$ —
maximum-inner-product search over a matrix that does **not** depend on the query and is therefore
precomputed offline. The MIPS order equals the full-scan order equals the cosine order (the unit
documents make dot and cosine agree), and query 0's top-1 is its own company.

In [ ]:
from the_retrieval_problem import dot, cosine, rank
corpus = {f"doc-{j}": P[j] for j in range(P.shape[0])}
ids, P_pre = dpr.precompute_passages(corpus)        # the offline document matrix
z_q = Q[0]
print("MIPS   top-k:", dpr.mips_retrieve(z_q, ids, P_pre, k=4))
print("scan   top-k:", rank(z_q, corpus, dot, descending=True)[:4])
print("cosine top-k:", rank(z_q, corpus, cosine, descending=True)[:4])
dpr.test_mips_reduction_equivalence()

## Movement 2 — The rank-$d$ expressivity ceiling

Stack the scores into $S=QG^\top$ with inner (embedding) dimension $d$. Then $\operatorname{rank}(S)\le d$,
and a target relevance matrix $M$ is realizable *exactly* by a $d$-dimensional dual encoder iff
$\operatorname{rank}(M)\le d$ — the thin-SVD construction. When $\operatorname{rank}(M)>d$, the best
realizable approximation in Frobenius norm is the truncated SVD (**Eckart–Young–Mirsky**, which we
cite rather than reprove).

The consequence for retrieval: the best rank-$d$ dual encoder's recall@1 **collapses** below the
relevance pattern's intrinsic rank and **saturates** above it.

In [ ]:
dpr.test_rank_d_realizability()
dpr.test_eckart_young_crosscheck()
curve = dpr.rank_recall_curve(S, truth)
for r in curve:
    print(f"  d = {r['d']}:  recall@1 = {r['r1']:.4f}   recall@3 = {r['r3']:.4f}   "
          f"recon-err = {r['recon_err']:.4f}")
print(f"recovers full recall@1 at d = {dpr.first_full_recall_dim(curve)} "
      f"(below rank = {P.shape[0]}); exact reconstruction at d = rank")
dpr.test_rank_ceiling_recall()

A cross-encoder $h([q;d])$ fuses the pair and is **not** a factorization, so it carries no rank
constraint — it can realize a full-rank relevance pattern a dual encoder cannot. This is the
expressivity price of separability (the deep cross-encoder treatment is its own topic).

In [ ]:
dpr.test_cross_encoder_has_no_rank_constraint()

## Movement 3 — In-batch negatives = the $B^2$-from-$2B$ Gram trick

For a batch of $B$ (query, positive) pairs, the $B\times B$ Gram matrix $S=QG^\top$ has the
positives on its diagonal; each row is an InfoNCE problem with the other $B-1$ documents as
negatives. So $2B$ encoder passes yield $B(B-1)$ negative comparisons. The loss is the **imported**
`info_nce_loss_batch` — we only re-read the Gram matrix the architecture produces, and assert
byte-for-byte agreement.

In [ ]:
dpr.test_inbatch_equals_imported_infonce()
dpr.test_counting_law()
dpr.test_symmetric_loss()
for B in dpr.BATCH_GRID:
    passes, negs = dpr.negative_pair_count(B)
    print(f"  B = {B:>3}:  {passes:>4} encoder passes  ->  {negs:>5} in-batch negatives")

## Finance case study and the viz constants

The synthetic finance dual encoder (reused vMF sectors$\to$companies geometry) ties the three
movements together, and `viz_constants()` prints exactly what `DenseDualEncoderLaboratory.tsx`
mirrors to the decimal.

In [ ]:
dpr.test_finance_dataset_is_reused()
dpr.finance_demo()
dpr.viz_constants()